# 02 — Train & Evaluate on Google Colab (LineSight)

**Visual Detection of Product Assembly Errors — MVTec LOCO AD**

⚠️ **Run every cell in order, top to bottom (Shift+Enter). Do not skip any.**
Each step checks the previous one and stops with a clear message if something
is missing — so you can't silently end up with no model.

First: enable the GPU → **Runtime → Change runtime type → T4 GPU → Save**.

## Step 1 — Get the project code

Set your GitHub repo URL below, then run the cell. It clones the repo and
moves into it. (If you renamed the repo, change the folder name too.)

In [ ]:
# 👉 EDIT THIS LINE: paste your repository URL
REPO_URL = 'https://github.com/YOUR-USERNAME/LineSight.git'

import os
REPO_DIR = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')

# Clone only if not already cloned (so re-running is safe).
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL

assert os.path.isdir(REPO_DIR), (
    f'Clone failed. Check REPO_URL is correct and the repo is public, '
    f'or use the no-GitHub method in Step 1b.')
%cd $REPO_DIR

# Hard check: the training code must be here.
assert os.path.exists('src/training/train_anomaly.py'), (
    'src/training/train_anomaly.py not found — you are not inside the project '
    'folder. Check the clone worked and that %cd moved into the repo.')
print('✅ Code is ready. Current folder:', os.getcwd())
!ls

### Step 1b — (ONLY if you have no GitHub repo) upload the project zip instead

Skip this if Step 1 worked. Otherwise: comment out Step 1, then uncomment and
run the lines below to upload `linesight.zip` from your computer.

In [ ]:
# from google.colab import files
# up = files.upload()                 # pick linesight.zip
# name = next(iter(up))
# !unzip -q "$name"
# %cd ivp
# assert __import__('os').path.exists('src/training/train_anomaly.py')
# print('✅ Project unzipped and ready.')

## Step 2 — Install dependencies

Colab already has torch + torchvision + matplotlib; we add the small extras.

In [ ]:
!pip install -q numpy Pillow
import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available(),
      '->', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
if not torch.cuda.is_available():
    print('⚠️  No GPU detected. Enable it: Runtime → Change runtime type → T4 GPU.')

## Step 3 — Upload the dataset

The dataset is **not** on GitHub (it is several GB), so you upload one category.

**On your PC first**, create the zip in PowerShell (from the project folder):
```powershell
Compress-Archive -Path "data\mvtec_loco\screw_bag" -DestinationPath "screw_bag.zip"
```
Then run the cell below and pick `screw_bag.zip` when the upload button appears.

In [ ]:
from google.colab import files
uploaded = files.upload()            # choose screw_bag.zip
zip_name = next(iter(uploaded))
print('Uploaded:', zip_name)

!mkdir -p /content/dataset
!unzip -q -o "$zip_name" -d /content/dataset
print('Unzipped into /content/dataset')

## Step 4 — Locate the dataset and point the project at it

This auto-detects where `screw_bag/train/good` landed (no matter how the zip was
nested) and sets the environment variables the scripts read. **It stops here if**
**the images aren't found** — so you never train on an empty folder.

In [ ]:
import os, glob

CATEGORY = 'screw_bag'
matches = glob.glob(f'/content/dataset/**/{CATEGORY}/train/good', recursive=True)
assert matches, (
    f'Could not find {CATEGORY}/train/good under /content/dataset. '
    f'Re-check the zip you uploaded actually contains the {CATEGORY} folder.')

train_good = matches[0]
data_root = train_good[: -len(f'/{CATEGORY}/train/good')]
n_train = len([f for f in os.listdir(train_good) if f.lower().endswith(('.png','.jpg','.jpeg','.bmp'))])
assert n_train > 0, f'{train_good} exists but is empty.'

os.environ['IVP_DATA_DIR'] = data_root
os.environ['IVP_CATEGORY'] = CATEGORY
os.environ['IVP_MODEL_BACKEND'] = 'autoencoder'
print('✅ Dataset root :', data_root)
print(f'✅ {CATEGORY}/train/good has {n_train} images')

## Step 5 — Build the manifest (dataset → CSV)

Scans the folders and lists every image with its split + label. You should see
counts like `train/good: 360, val/good: 60, test/...`.

In [ ]:
!python -m src.training.prepare_data --category $IVP_CATEGORY

## Step 6 — Train the autoencoder

Trains on `train/good` only, tracks train + validation loss, saves a **training
curve**, and calibrates the PASS/FAIL threshold. A few minutes on a T4.

Watch for the final lines: `Training curve saved -> ...` and `Saved model -> ...`.

In [ ]:
!python -m src.training.train_anomaly --category $IVP_CATEGORY --epochs 30

### Confirm the training actually produced files

This must list a `.pt` model and the training-curve PNG. If it's empty, the cell
above failed — scroll up and read its error, don't continue.

In [ ]:
import os
ok_model = os.path.exists(f'artifacts/models/autoencoder_{os.environ["IVP_CATEGORY"]}.pt')
ok_curve = os.path.exists(f'artifacts/eval/training_curve_{os.environ["IVP_CATEGORY"]}.png')
print('model saved :', ok_model)
print('curve saved :', ok_curve)
assert ok_model, 'No model file — training did not finish. Read the error above.'
!ls -la artifacts/models artifacts/eval

### View the training curve

Train and validation loss should both go down and stay close (no overfitting).

In [ ]:
from IPython.display import Image as IPyImage
import os
curve = f"artifacts/eval/training_curve_{os.environ['IVP_CATEGORY']}.png"
IPyImage(filename=curve) if os.path.exists(curve) else print('Curve not found — re-run Step 6.')

## Step 7 — Evaluate on the test set

Prints the confusion matrix, precision / recall / F1, accuracy and AUROC, and
saves confusion + ROC plots. **Copy these numbers into your report (section 4).**

In [ ]:
!python -m src.training.evaluate --category $IVP_CATEGORY

In [ ]:
from IPython.display import Image as IPyImage, display
import os
cat = os.environ['IVP_CATEGORY']
for p in (f'artifacts/eval/confusion_{cat}.png', f'artifacts/eval/roc_{cat}.png'):
    if os.path.exists(p):
        display(IPyImage(filename=p))

## Step 8 — Download the trained model for your local demo

Saves the model + calibration so you can run the Streamlit app on your PC without
retraining. Put both downloaded files into your local `artifacts/models/` folder.

In [ ]:
from google.colab import files
import os
cat = os.environ['IVP_CATEGORY']
files.download(f'artifacts/models/autoencoder_{cat}.pt')
files.download(f'artifacts/models/autoencoder_{cat}.json')

## ✅ Done

You now have: a manifest (data integration), a trained model + training curve,
and evaluation metrics with confusion matrix + ROC.

**On your PC**, drop the two downloaded files into `artifacts/models/`, then:
```powershell
pip install -r requirements.txt
$env:IVP_MODEL_BACKEND = 'autoencoder'
streamlit run src/app/streamlit_app.py
```
Your demo now runs on the real trained model. 🎯